# 3.15 — Softmax (Multinomial) Regression

Softmax regression turns linear class scores into a probability distribution, then learns weights by minimizing average cross-entropy plus any cost or regularization we decide belongs in the selection score. In this lesson, you will build the probability formula, the loss, the gradient, and the validation-style decision arithmetic from scratch with NumPy so every number behind the model is inspectable.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build softmax regression one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math, including the stable normalization, cross-entropy, gradient, and final model-selection score, is derived and shown. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays + linear algebra for scores, probabilities, and gradients.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for toy points and training loops.

### 1. From feature vectors to one score per class

Multinomial regression starts by assigning each class its own linear score. For an input vector $x$, the model stores one weight vector per class in $W$ and one bias per class in $b$, then computes $z=xW+b$. The scores are not probabilities yet; they are comparable evidence numbers, and shifting all of them by the same constant should not change the final class belief.

In [ ]:
x_w = np.array([1.0, 2.0])  # one example with two features.
W_w = np.array([[0.40, -0.20, 0.10],
                [0.30,  0.50, -0.40]])  # two features by three classes.
b_w = np.array([0.10, -0.20, 0.30])  # one intercept per class.
z_w = x_w @ W_w + b_w  # linear scores for the three possible classes.
print("scores z:", np.round(z_w, 3))
assert np.allclose(np.round(z_w, 3), [1.1, 0.6, -0.4])

▶ What you'll see: class 0 has the largest raw score, class 1 is close behind, and class 2 is far lower.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["class 0", "class 1", "class 2"], z_w, color="steelblue")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("1: linear class scores before softmax")
plt.ylabel("score z_k")
plt.show()

▶ What you'll see: a bar chart of unnormalized evidence; these numbers can be negative and need not sum to one.

*Why it's done this way:* a separate linear score per class lets each class define its own direction in feature space. We delay probability normalization because linear algebra is the simple part; softmax will later make the scores comparable on a shared probability scale without changing their order.

### 2. Softmax turns scores into probabilities

Softmax exponentiates each score and divides by the sum of all exponentials:

$$p_k=\frac{e^{z_k}}{\sum_j e^{z_j}}.$$

Exponentials make every term positive, and the denominator forces the outputs to sum to one. Larger scores get larger probabilities, but every class keeps some probability unless its score is infinitely worse.

In [ ]:
exp_w = np.exp(z_w)  # positive evidence for each class.
p_w = exp_w / np.sum(exp_w)  # normalize evidence into probabilities.
print("exp(z):", np.round(exp_w, 3))
print("softmax probabilities:", np.round(p_w, 3), "sum:", round(float(np.sum(p_w)), 3))
assert np.allclose(np.round(p_w, 3), [0.547, 0.331, 0.122])
assert round(float(np.sum(p_w)), 6) == 1.0

▶ What you'll see: the probabilities sum to exactly 1, with the largest score receiving about 54.7% probability.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["class 0", "class 1", "class 2"], p_w, color="seagreen")
plt.ylim(0, 1)
plt.title("2: softmax probabilities")
plt.ylabel("p_k")
plt.show()

▶ What you'll see: a proper categorical distribution; the bars are now probabilities rather than arbitrary scores.

*Why it's done this way:* exponentiation converts score gaps into evidence ratios: if one score is 1 unit larger, its unnormalized evidence is $e$ times larger. Dividing by the total evidence is the only step that makes the class probabilities mutually exclusive and collectively exhaustive.

### 3. Numerical stability: subtract the maximum score

Large scores can overflow when exponentiated, even though softmax itself is unchanged by adding or subtracting the same constant from every score. The stable trick is to subtract $\max_k z_k$ before exponentiating. This leaves all probability ratios identical but makes the largest shifted score equal to 0, so its exponential is safely 1.

In [ ]:
big_z_w = np.array([1001.1, 1000.6, 999.6])  # same score gaps as z_w, shifted upward by 1000.
shifted_w = big_z_w - np.max(big_z_w)  # stable scores with the same differences.
p_stable_w = np.exp(shifted_w) / np.sum(np.exp(shifted_w))
print("shifted scores:", np.round(shifted_w, 3))
print("stable softmax:", np.round(p_stable_w, 3))
assert np.allclose(np.round(p_stable_w, 3), [0.547, 0.331, 0.122])

▶ What you'll see: subtracting the maximum turns huge scores into `[0, -0.5, -1.5]` and recovers the same probabilities.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.plot(["raw class0", "raw class1", "raw class2"], big_z_w, marker="o", label="huge scores")
plt.plot(["raw class0", "raw class1", "raw class2"], shifted_w, marker="s", label="shifted scores")
plt.title("3: same gaps, safer exponentials")
plt.ylabel("score value")
plt.legend()
plt.show()

▶ What you'll see: the shifted scores have the same spacing but are numerically safe to exponentiate.

*Why it's done this way:* softmax depends only on score differences because the common factor $e^c$ cancels from numerator and denominator. Subtracting the maximum uses that invariance to prevent overflow without changing the mathematics.

### 4. Cross-entropy loss for the true class

For a one-hot label $y$, softmax regression uses negative log likelihood: $L=-\log p_y$. If the model assigns high probability to the correct class, the loss is small; if it assigns tiny probability, the logarithm creates a large penalty. This is the per-example loss that later gets averaged as empirical risk.

In [ ]:
y_w = 0  # the true class for this example.
loss_w = -np.log(p_w[y_w])
print("true-class probability:", round(float(p_w[y_w]), 3))
print("cross-entropy loss:", round(float(loss_w), 3))
assert round(float(loss_w), 3) == 0.604

▶ What you'll see: class 0 was likely but not certain, so the loss is moderate rather than zero.

In [ ]:
prob_grid_w = np.linspace(0.01, 0.99, 100)
loss_grid_w = -np.log(prob_grid_w)
plt.figure(figsize=(4.4, 3))
plt.plot(prob_grid_w, loss_grid_w, color="purple")
plt.scatter([p_w[y_w]], [loss_w], color="red")
plt.title("4: -log(probability of the true class)")
plt.xlabel("p_true")
plt.ylabel("loss")
plt.show()

▶ What you'll see: loss falls quickly as the correct-class probability approaches 1 and explodes near 0.

*Why it's done this way:* the log loss is exactly the cost of saying how surprising the true label was under the model's probability distribution. It rewards calibrated confidence, not just the top-1 class, because changing 0.55 to 0.90 for the correct label genuinely lowers the loss.

### 5. Empirical risk averages losses over examples

The lesson's verified toy losses are 0.246, 0.122, and 0.488. Empirical risk is just their average: the model is judged by its typical training loss, not by its best example or worst example alone.

In [ ]:
losses_w = np.array([0.246, 0.122, 0.488])  # verified per-example losses from the lesson text.
R_S_w = float(np.mean(losses_w))
print("loss sum:", round(float(np.sum(losses_w)), 3))
print("empirical risk R_S:", round(R_S_w, 3))
assert round(float(np.sum(losses_w)), 3) == 0.856
assert round(R_S_w, 3) == 0.285

▶ What you'll see: `(0.246 + 0.122 + 0.488) / 3 = 0.285` after rounding.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["ex 1", "ex 2", "ex 3"], losses_w, color="darkorange")
plt.axhline(R_S_w, color="black", linestyle="--", label=f"mean={R_S_w:.3f}")
plt.title("5: empirical risk averages losses")
plt.ylabel("cross-entropy loss")
plt.legend()
plt.show()

▶ What you'll see: the mean line summarizes the three per-example losses into the quantity optimized by ERM.

*Why it's done this way:* averaging is what makes training size comparable. A sum would grow whenever we add examples; the average estimates future expected loss on the same scale as each individual example.

### 6. The gradient has the simple form p minus y

For one example, the gradient of cross-entropy with softmax scores is $p-y$, where $y$ is a one-hot vector. If the correct class probability is too low, its component is negative, so gradient descent increases that score. Incorrect classes have positive components, so descent lowers their scores.

In [ ]:
y_onehot_w = np.array([1.0, 0.0, 0.0])
grad_z_w = p_w - y_onehot_w
print("p - y:", np.round(grad_z_w, 3))
print("gradient sums to:", round(float(np.sum(grad_z_w)), 6))
assert np.allclose(np.round(grad_z_w, 3), [-0.453, 0.331, 0.122])
assert round(float(np.sum(grad_z_w)), 6) == 0.0

▶ What you'll see: the true class gets a negative gradient and the other classes get positive gradients.

In [ ]:
grad_W_w = np.outer(x_w, grad_z_w)  # feature-by-class weight gradient.
print("weight gradient:\n", np.round(grad_W_w, 3))
plt.figure(figsize=(4.6, 3.2))
plt.imshow(grad_W_w, cmap="coolwarm", aspect="auto")
plt.colorbar(label="gradient")
plt.title("6: ∂L/∂W = xᵀ(p-y)")
plt.xlabel("class")
plt.ylabel("feature")
plt.show()

▶ What you'll see: feature 2 has twice the magnitude of feature 1 because the input value was 2 instead of 1.

*Why it's done this way:* softmax and log loss simplify beautifully: all the complicated normalization collapses into prediction minus target. The outer product then says each feature receives credit or blame in proportion to how present that feature was in the example.

### 7. One gradient-descent step moves probability toward the label

Gradient descent updates $W\leftarrow W-\eta\nabla W$ and $b\leftarrow b-\eta\nabla b$. Because the correct-class score gradient is negative, subtracting it raises the true class score; subtracting positive gradients lowers competing scores.

In [ ]:
eta_w = 0.2
W_new_w = W_w - eta_w * grad_W_w
b_new_w = b_w - eta_w * grad_z_w
z_new_w = x_w @ W_new_w + b_new_w
p_new_w = np.exp(z_new_w - np.max(z_new_w)) / np.sum(np.exp(z_new_w - np.max(z_new_w)))
print("old p_true:", round(float(p_w[y_w]), 3), "new p_true:", round(float(p_new_w[y_w]), 3))
assert round(float(p_new_w[y_w]), 3) == 0.742

▶ What you'll see: one step raises the correct class probability from 0.547 to about 0.742.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["before", "after"], [p_w[y_w], p_new_w[y_w]], color=["gray", "seagreen"])
plt.ylim(0, 1)
plt.title("7: one step raises p(true class)")
plt.ylabel("probability of class 0")
plt.show()

▶ What you'll see: the update moves the probability mass toward the observed label.

*Why it's done this way:* the gradient is local evidence about which score changes reduce the current example's loss. Taking a small step uses that evidence without assuming the one example should completely rewrite the model.

### 8. Add cost or regularization before choosing a model

The source lesson warns that raw training loss is not the full decision score. Here the cost is 0.080, so the selection score is $R_S+cost=0.365$. This is the same logic as regularization: flexible models must pay for complexity before we trust their training fit.

In [ ]:
cost_w = 0.080
score_w = round(R_S_w + cost_w, 3)
print("raw R_S:", round(R_S_w, 3), "cost:", round(cost_w, 3), "score:", round(score_w, 3))
assert round(score_w, 3) == 0.365

▶ What you'll see: the decision score is higher than the raw empirical risk because it includes the method's cost.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["R_S", "cost", "R_S + cost"], [R_S_w, cost_w, score_w], color=["teal", "orange", "purple"])
plt.title("8: selection uses the full score")
plt.ylabel("score component")
plt.show()

▶ What you'll see: the full bar is not just the training average; the cost term visibly changes what is optimized.

*Why it's done this way:* a low training loss can come from fitting noise. Adding cost or regularization encodes the belief that simpler or more stable rules deserve preference unless flexibility earns its keep.

### 9. Compare alternatives with absolute and relative gaps

A more flexible alternative reaches decision score 0.401. The baseline score 0.365 is lower, but the absolute gap 0.036 and relative gap about 0.090 tell us how strong that preference is. A tiny gap may vanish under resampling noise, so the gap is evidence, not decoration.

In [ ]:
alt_score_w = 0.401
gap_w = alt_score_w - score_w
relative_gap_w = gap_w / alt_score_w
print("gap:", round(gap_w, 3))
print("relative gap:", round(relative_gap_w, 3))
assert round(gap_w, 3) == 0.036
assert round(relative_gap_w, 3) == 0.090

▶ What you'll see: the baseline beats the flexible alternative by 0.036, about 9.0% of the alternative's score.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["baseline", "flexible alt"], [score_w, alt_score_w], color=["seagreen", "crimson"])
plt.ylabel("lower is better")
plt.title("9: compare full decision scores")
plt.show()

▶ What you'll see: the lower bar wins, but the visual gap is modest rather than overwhelming.

*Why it's done this way:* model selection is a comparison on a shared scale. Reporting the gap prevents us from pretending that a narrow numerical win is as trustworthy as a large one.

### 10. Stabilization and the end-to-end decision

If a stabilizing knob reduces the decision score by 20%, the new score is $0.80\cdot0.365=0.292$. The final decision compares the baseline, flexible alternative, and stabilized score, then carries forward the lowest full score.

In [ ]:
stable_w = 0.80 * score_w
candidates_w = np.array([score_w, alt_score_w, stable_w])
labels_w = np.array(["baseline", "flexible", "stabilized"])
best_w = labels_w[int(np.argmin(candidates_w))]
print("scores:", np.round(candidates_w, 3))
print("best choice:", best_w)
assert round(stable_w, 3) == 0.292
assert best_w == "stabilized"

▶ What you'll see: the stabilized score is lowest, so it is the toy decision to carry forward.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.bar(labels_w, candidates_w, color=["gray", "crimson", "seagreen"])
plt.ylabel("full decision score")
plt.title("10: end-to-end model selection")
plt.show()

▶ What you'll see: the stabilized bar is the minimum among the three candidate decision scores.

*Why it's done this way:* the final model is chosen by the full objective implied by the method, not by whichever fragment looks most flattering. Stabilization does not always win, but this arithmetic shows exactly what would have to be true for it to win here.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, vectorized scores, probabilities, losses, and gradients.
import matplotlib.pyplot as plt # load Matplotlib so every concept can be inspected visually.
np.random.seed(0) # make all random examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Compute three linear class scores

**Goal.** Turn one feature vector into one score per class, because softmax regression begins with linear evidence before probability normalization. We build it in 2 steps.

In [ ]:
x_b1 = np.array([1.0, 2.0]) # store one two-feature example.
W_b1 = np.array([[0.40, -0.20, 0.10], [0.30, 0.50, -0.40]]) # store one weight vector per class as columns.
b_b1 = np.array([0.10, -0.20, 0.30]) # store one intercept per class.
print("x shape:", x_b1.shape, "W shape:", W_b1.shape, "b shape:", b_b1.shape) # inspect dimensions before multiplication.

In [ ]:
z_b1 = x_b1 @ W_b1 + b_b1 # compute the three class scores z = xW + b.
print("scores:", np.round(z_b1, 3)) # inspect raw class evidence.
assert np.allclose(np.round(z_b1, 3), [1.1, 0.6, -0.4]) # verify the worked score vector.
plt.figure(figsize=(4, 3)) # create a compact score plot.
plt.bar(["c0", "c1", "c2"], z_b1, color="steelblue") # visualize raw evidence per class.
plt.axhline(0, color="black", linewidth=0.8) # show the zero-score reference line.
plt.title("Basic 1: class scores") # title the figure with the example id.
plt.ylabel("z_k") # label the raw score scale.
plt.show() # display the bar chart.

▶ What you'll see: class 0 has the highest raw evidence, but the numbers are not probabilities yet.

👀 Takeaway: multinomial regression creates one linear score for each possible class.

### Basic 2 — Exponentiate the scores

**Goal.** Convert arbitrary scores into positive evidence, because probabilities cannot be negative and softmax compares evidence ratios. We build it in 2 steps.

In [ ]:
z_b2 = np.array([1.1, 0.6, -0.4]) # reuse the worked class scores.
exp_b2 = np.exp(z_b2) # exponentiate each score to get positive unnormalized evidence.
print("exp(z):", np.round(exp_b2, 3)) # inspect the positive evidence values.

In [ ]:
ratios_b2 = exp_b2 / exp_b2[1] # compare every class's evidence against class 1.
print("evidence ratios vs class 1:", np.round(ratios_b2, 3)) # inspect how score gaps become multiplicative ratios.
assert round(float(ratios_b2[0]), 3) == 1.649 # verify e^(1.1-0.6).
plt.figure(figsize=(4, 3)) # create a compact evidence plot.
plt.bar(["c0", "c1", "c2"], exp_b2, color="orange") # visualize unnormalized positive evidence.
plt.title("Basic 2: exponentiated scores") # title the plot.
plt.ylabel("e^{z_k}") # label the exponential evidence scale.
plt.show() # display the evidence bars.

▶ What you'll see: class 0 has about 1.65 times class 1's evidence because its score is 0.5 higher.

👀 Takeaway: exponentials turn additive score gaps into multiplicative evidence ratios.

### Basic 3 — Normalize with softmax

**Goal.** Divide positive evidence by total evidence, because a multiclass prediction must be a probability distribution that sums to one. We build it in 2 steps.

In [ ]:
z_b3 = np.array([1.1, 0.6, -0.4]) # define three class scores.
exp_b3 = np.exp(z_b3) # compute positive evidence values.
total_b3 = np.sum(exp_b3) # compute the denominator shared by all classes.
print("total evidence:", round(float(total_b3), 3)) # inspect the normalization constant.

In [ ]:
p_b3 = exp_b3 / total_b3 # normalize into probabilities.
print("probabilities:", np.round(p_b3, 3), "sum:", round(float(np.sum(p_b3)), 3)) # inspect probabilities and their sum.
assert np.allclose(np.round(p_b3, 3), [0.547, 0.331, 0.122]) # verify the canonical probabilities.
plt.figure(figsize=(4, 3)) # create a compact probability chart.
plt.bar(["c0", "c1", "c2"], p_b3, color="seagreen") # draw one probability bar per class.
plt.ylim(0, 1) # keep the probability scale visible.
plt.title("Basic 3: softmax probabilities") # title the plot.
plt.ylabel("p_k") # label the probability axis.
plt.show() # display the chart.

▶ What you'll see: all probabilities are positive and sum to 1.

👀 Takeaway: softmax is exponentiate-then-normalize for mutually exclusive classes.

### Basic 4 — Predict the most likely class

**Goal.** Convert a probability vector into a predicted label, because classification uses the class with the largest probability unless a downstream rule says otherwise. We build it in 2 steps.

In [ ]:
p_b4 = np.array([0.547, 0.331, 0.122]) # store a softmax probability vector.
pred_class_b4 = int(np.argmax(p_b4)) # choose the index with the largest probability.
print("predicted class:", pred_class_b4) # inspect the class decision.
assert pred_class_b4 == 0 # verify class 0 wins.

In [ ]:
margin_b4 = float(np.sort(p_b4)[-1] - np.sort(p_b4)[-2]) # compute top probability minus runner-up.
print("top-vs-runner-up margin:", round(margin_b4, 3)) # inspect confidence gap.
plt.figure(figsize=(4, 3)) # create a compact prediction plot.
plt.bar(["c0", "c1", "c2"], p_b4, color=["seagreen", "gray", "gray"]) # highlight the winning class.
plt.title("Basic 4: argmax prediction") # title the plot.
plt.ylabel("probability") # label the probability scale.
plt.show() # display the bars.

▶ What you'll see: class 0 wins, but class 1 is close enough that the margin is worth inspecting.

👀 Takeaway: argmax gives the predicted class, while the probability gap shows how decisive the win is.

### Basic 5 — Compute one cross-entropy loss

**Goal.** Penalize the model according to the probability assigned to the true class, because softmax regression is trained by negative log likelihood. We build it in 2 steps.

In [ ]:
p_b5 = np.array([0.547, 0.331, 0.122]) # define predicted class probabilities.
y_b5 = 0 # set the true class.
true_prob_b5 = p_b5[y_b5] # pick out the probability assigned to the true label.
print("true-class probability:", true_prob_b5) # inspect the quantity that enters the log loss.

In [ ]:
loss_b5 = -np.log(true_prob_b5) # compute cross-entropy for one one-hot label.
print("loss:", round(float(loss_b5), 3)) # inspect the per-example loss.
assert round(float(loss_b5), 3) == 0.603 # verify the rounded lesson number.
plt.figure(figsize=(4, 3)) # create a compact diagnostic chart.
plt.bar(["p_true", "-log(p_true)"], [true_prob_b5, loss_b5], color=["teal", "purple"]) # compare probability and loss.
plt.title("Basic 5: true-class log loss") # title the plot.
plt.show() # display the chart.

▶ What you'll see: a probability a bit above 0.5 produces a moderate loss around 0.603.

👀 Takeaway: cross-entropy gets small only when the true class receives high probability.

### Basic 6 — Average per-example losses

**Goal.** Compute empirical risk from three verified losses, because ERM optimizes an average training loss. We build it in 2 steps.

In [ ]:
losses_b6 = np.array([0.246, 0.122, 0.488]) # store the verified toy losses from the lesson source.
sum_b6 = float(np.sum(losses_b6)) # add the three losses.
print("loss sum:", round(sum_b6, 3)) # inspect the numerator of the average.
assert round(sum_b6, 3) == 0.856 # verify the arithmetic used in the source lesson.

In [ ]:
risk_b6 = float(np.mean(losses_b6)) # compute empirical risk as the average loss.
print("R_S:", round(risk_b6, 3)) # inspect the training risk.
assert round(risk_b6, 3) == 0.285 # verify 0.856/3 rounded.
plt.figure(figsize=(4, 3)) # create a compact loss chart.
plt.bar(["ex1", "ex2", "ex3"], losses_b6, color="darkorange") # show per-example losses.
plt.axhline(risk_b6, color="black", linestyle="--") # show the average risk.
plt.title("Basic 6: average training loss") # title the plot.
plt.ylabel("loss") # label the loss scale.
plt.show() # display the chart.

▶ What you'll see: one average line summarizes the three training losses.

👀 Takeaway: empirical risk is the mean loss over the training sample.

### Basic 7 — Add a cost term

**Goal.** Add the method cost to raw empirical risk, because the source lesson selects models with the full decision score rather than training loss alone. We build it in 2 steps.

In [ ]:
risk_b7 = 0.285 # use the rounded empirical risk from the verified toy arithmetic.
cost_b7 = 0.080 # define the complexity, regularization, or operational cost.
score_b7 = risk_b7 + cost_b7 # compute the full selection score.
print("score:", round(score_b7, 3)) # inspect the decision quantity.
assert round(score_b7, 3) == 0.365 # verify the source lesson score.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact component chart.
plt.bar(["risk", "cost", "score"], [risk_b7, cost_b7, score_b7], color=["teal", "orange", "purple"]) # compare raw and penalized quantities.
plt.title("Basic 7: risk plus cost") # title the plot.
plt.ylabel("value") # label the numeric scale.
plt.show() # display the chart.

▶ What you'll see: the decision score is visibly larger than the raw training risk.

👀 Takeaway: model selection should include the cost or regularization term promised by the objective.

### Basic 8 — Compare a flexible alternative

**Goal.** Compute absolute and relative score gaps, because a model-selection win should be read as evidence with a size. We build it in 2 steps.

In [ ]:
baseline_b8 = 0.365 # full score for the baseline setting.
alternative_b8 = 0.401 # full score for the more flexible alternative.
gap_b8 = alternative_b8 - baseline_b8 # compute lower-is-better advantage of the baseline.
print("gap:", round(gap_b8, 3)) # inspect absolute evidence.
assert round(gap_b8, 3) == 0.036 # verify the source lesson gap.

In [ ]:
relative_gap_b8 = gap_b8 / alternative_b8 # scale the gap by the alternative score.
print("relative gap:", round(relative_gap_b8, 3)) # inspect proportional evidence.
assert round(relative_gap_b8, 3) == 0.090 # verify the rounded relative gap.
plt.figure(figsize=(4, 3)) # create a compact comparison plot.
plt.bar(["baseline", "alternative"], [baseline_b8, alternative_b8], color=["seagreen", "crimson"]) # compare full scores.
plt.title("Basic 8: full-score comparison") # title the plot.
plt.ylabel("lower is better") # label the score axis.
plt.show() # display the chart.

▶ What you'll see: the baseline has the lower score, with about a 9% relative gap.

👀 Takeaway: score gaps tell you how meaningful a numerical win might be.

### Basic 9 — Apply a stabilization multiplier

**Goal.** Compute the stabilized score, because the source lesson shows how a stability knob can improve a decision score by reducing brittle variation. We build it in 2 steps.

In [ ]:
score_b9 = 0.365 # baseline full score before stabilization.
multiplier_b9 = 0.80 # a 20% reduction keeps 80% of the original score.
stable_b9 = multiplier_b9 * score_b9 # compute the stabilized score.
print("stabilized score:", round(stable_b9, 3)) # inspect the new score.
assert round(stable_b9, 3) == 0.292 # verify 0.80 * 0.365 rounded.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact before-after plot.
plt.bar(["before", "after stability"], [score_b9, stable_b9], color=["gray", "seagreen"]) # compare scores before and after the knob.
plt.title("Basic 9: stabilization lowers score") # title the plot.
plt.ylabel("full decision score") # label the score scale.
plt.show() # display the chart.

▶ What you'll see: the stabilized score is lower because the multiplier is less than one.

👀 Takeaway: constraints or regularization can win when their stability benefit outweighs lost flexibility.

### Basic 10 — Choose the lowest full score

**Goal.** Make the end-to-end decision among baseline, flexible, and stabilized scores, because the final model is selected by the full objective. We build it in 2 steps.

In [ ]:
scores_b10 = np.array([0.365, 0.401, 0.292]) # store baseline, flexible alternative, and stabilized scores.
labels_b10 = np.array(["baseline", "flexible", "stabilized"]) # label each candidate.
best_idx_b10 = int(np.argmin(scores_b10)) # find the lowest score.
print("best:", labels_b10[best_idx_b10], "score:", scores_b10[best_idx_b10]) # inspect the selected model.
assert labels_b10[best_idx_b10] == "stabilized" # verify the source lesson decision.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact model-selection chart.
plt.bar(labels_b10, scores_b10, color=["gray", "crimson", "seagreen"]) # draw all candidate scores.
plt.title("Basic 10: final selection") # title the plot.
plt.ylabel("lower is better") # label the score axis.
plt.show() # display the chart.

▶ What you'll see: the stabilized candidate has the shortest bar.

👀 Takeaway: the correct unit of judgment is the full score implied by the method.

## 🟡 Easy

### Easy 1 — Implement stable softmax for a batch

**Goal.** Convert a batch of score rows into probabilities, because real softmax regression predicts many examples at once. We build it in 3 steps.

In [ ]:
Z_e1 = np.array([[1.1, 0.6, -0.4], [0.2, 1.4, -0.1], [-0.6, 0.1, 1.2]]) # define three examples by three class scores.
shift_e1 = Z_e1 - np.max(Z_e1, axis=1, keepdims=True) # subtract each row's maximum for numerical stability.
print("shifted scores:\n", np.round(shift_e1, 3)) # inspect stable score rows.

In [ ]:
exp_e1 = np.exp(shift_e1) # exponentiate shifted scores.
P_e1 = exp_e1 / np.sum(exp_e1, axis=1, keepdims=True) # normalize each row independently.
print("probabilities:\n", np.round(P_e1, 3)) # inspect batch softmax output.
print("row sums:", np.round(np.sum(P_e1, axis=1), 3)) # verify each example sums to one.
assert np.allclose(np.sum(P_e1, axis=1), np.ones(3)) # self-check probability rows.

In [ ]:
plt.figure(figsize=(4.8, 3)) # create a compact heatmap for batch probabilities.
plt.imshow(P_e1, cmap="viridis", aspect="auto") # visualize probability mass by example and class.
plt.colorbar(label="probability") # add a color scale.
plt.title("Easy 1: batch softmax") # title the plot.
plt.xlabel("class") # label columns as classes.
plt.ylabel("example") # label rows as examples.
plt.show() # display the heatmap.

▶ What you'll see: each row is its own probability distribution, and the brightest cell marks the predicted class.

👀 Takeaway: stable softmax subtracts a row-wise maximum before exponentiating.

### Easy 2 — Compute multiclass cross-entropy for a batch

**Goal.** Average negative log probabilities over labeled examples, because softmax regression minimizes empirical cross-entropy. We build it in 3 steps.

In [ ]:
P_e2 = np.array([[0.547, 0.331, 0.122], [0.202, 0.669, 0.129], [0.111, 0.224, 0.665]]) # store batch probabilities.
y_e2 = np.array([0, 1, 2]) # store one true class per example.
true_probs_e2 = P_e2[np.arange(len(y_e2)), y_e2] # gather probabilities assigned to true classes.
print("true probabilities:", true_probs_e2) # inspect selected probabilities.

In [ ]:
losses_e2 = -np.log(true_probs_e2) # compute one cross-entropy per example.
risk_e2 = float(np.mean(losses_e2)) # average losses into empirical risk.
print("losses:", np.round(losses_e2, 3)) # inspect per-example penalties.
print("mean loss:", round(risk_e2, 3)) # inspect empirical risk.
assert round(risk_e2, 3) == 0.471 # verify the rounded batch average.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact loss plot.
plt.bar(["ex0", "ex1", "ex2"], losses_e2, color="purple") # visualize per-example losses.
plt.axhline(risk_e2, color="black", linestyle="--", label="mean") # show the empirical average.
plt.title("Easy 2: batch cross-entropy") # title the plot.
plt.ylabel("loss") # label the loss scale.
plt.legend() # show the mean label.
plt.show() # display the chart.

▶ What you'll see: examples with lower true-class probability have larger negative-log loss.

👀 Takeaway: batch cross-entropy is the mean of true-class surprise values.

### Easy 3 — Train a tiny softmax classifier

**Goal.** Fit weights with gradient descent, because softmax regression learns score directions from labeled data rather than hand-coded scores. We build it in 4 steps.

In [ ]:
X_e3 = np.array([[2.0, 0.2], [1.7, -0.1], [-0.2, 1.8], [0.1, 2.1], [-1.8, -1.1], [-2.1, -0.8]]) # six points in three clusters.
y_e3 = np.array([0, 0, 1, 1, 2, 2]) # assign two examples to each class.
W_e3 = np.zeros((2, 3)) # initialize class weights at zero.
b_e3 = np.zeros(3) # initialize class biases at zero.
print("X shape:", X_e3.shape, "classes:", np.unique(y_e3)) # inspect the toy dataset.

In [ ]:
losses_e3 = [] # store the loss curve for plotting.
for step_e3 in range(300): # run batch gradient descent.
    Z_e3 = X_e3 @ W_e3 + b_e3 # compute class scores for all examples.
    Zs_e3 = Z_e3 - np.max(Z_e3, axis=1, keepdims=True) # stabilize scores row by row.
    P_e3 = np.exp(Zs_e3) / np.sum(np.exp(Zs_e3), axis=1, keepdims=True) # compute probabilities.
    Y_e3 = np.eye(3)[y_e3] # make one-hot labels.
    loss_e3 = -np.mean(np.log(P_e3[np.arange(len(y_e3)), y_e3])) # compute mean cross-entropy.
    losses_e3.append(loss_e3) # record the current loss.
    G_e3 = (P_e3 - Y_e3) / len(y_e3) # compute score gradients averaged over examples.
    W_e3 -= 0.4 * (X_e3.T @ G_e3) # update weights by gradient descent.
    b_e3 -= 0.4 * np.sum(G_e3, axis=0) # update biases by gradient descent.
print("loss start -> end:", round(losses_e3[0], 3), "->", round(losses_e3[-1], 3)) # inspect convergence.
assert losses_e3[-1] < 0.05 # verify the tiny separable dataset was learned.

In [ ]:
pred_e3 = np.argmax(X_e3 @ W_e3 + b_e3, axis=1) # predict classes from trained scores.
acc_e3 = float(np.mean(pred_e3 == y_e3)) # compute training accuracy.
print("predictions:", pred_e3, "accuracy:", acc_e3) # inspect final classification.
assert acc_e3 == 1.0 # verify all six toy points are classified correctly.

In [ ]:
plt.figure(figsize=(4.8, 3)) # create a compact convergence plot.
plt.plot(losses_e3, color="teal") # draw loss over gradient steps.
plt.title("Easy 3: softmax training loss") # title the plot.
plt.xlabel("step") # label the optimization step axis.
plt.ylabel("cross-entropy") # label the loss axis.
plt.show() # display the learning curve.

▶ What you'll see: loss drops steadily and the final training predictions match all six labels.

👀 Takeaway: the gradient `P - Y` is enough to train a multiclass linear classifier.

### Easy 4 — Visualize learned decision regions

**Goal.** Plot the trained classifier over a grid, because softmax regression creates linear score boundaries between classes. We build it in 4 steps.

In [ ]:
X_e4 = np.array([[2.0, 0.2], [1.7, -0.1], [-0.2, 1.8], [0.1, 2.1], [-1.8, -1.1], [-2.1, -0.8]]) # recreate the three-cluster dataset.
y_e4 = np.array([0, 0, 1, 1, 2, 2]) # recreate labels.
W_e4 = np.zeros((2, 3)) # initialize weights for a fresh self-contained example.
b_e4 = np.zeros(3) # initialize biases.
print("training points:", len(y_e4)) # inspect dataset size.

In [ ]:
for step_e4 in range(300): # train a small softmax model.
    Z_e4 = X_e4 @ W_e4 + b_e4 # compute current scores.
    P_e4 = np.exp(Z_e4 - np.max(Z_e4, axis=1, keepdims=True)) # exponentiate stable scores.
    P_e4 = P_e4 / np.sum(P_e4, axis=1, keepdims=True) # normalize into probabilities.
    G_e4 = (P_e4 - np.eye(3)[y_e4]) / len(y_e4) # compute average score gradients.
    W_e4 -= 0.4 * (X_e4.T @ G_e4) # update class weights.
    b_e4 -= 0.4 * np.sum(G_e4, axis=0) # update class biases.
print("trained W:\n", np.round(W_e4, 3)) # inspect learned class directions.

In [ ]:
grid_x_e4, grid_y_e4 = np.meshgrid(np.linspace(-3, 3, 80), np.linspace(-2.5, 3, 80)) # create grid coordinates.
grid_e4 = np.c_[grid_x_e4.ravel(), grid_y_e4.ravel()] # flatten grid into examples.
region_e4 = np.argmax(grid_e4 @ W_e4 + b_e4, axis=1).reshape(grid_x_e4.shape) # classify every grid point.
print("grid shape:", region_e4.shape) # inspect the prediction grid.

In [ ]:
plt.figure(figsize=(5, 4)) # create a decision-region figure.
plt.contourf(grid_x_e4, grid_y_e4, region_e4, levels=[-0.5, 0.5, 1.5, 2.5], alpha=0.25, colors=["tab:blue", "tab:orange", "tab:green"]) # shade predicted class regions.
plt.scatter(X_e4[:, 0], X_e4[:, 1], c=y_e4, cmap="viridis", edgecolor="black", s=70) # overlay training points.
plt.title("Easy 4: learned softmax regions") # title the plot.
plt.xlabel("feature 0") # label the first feature axis.
plt.ylabel("feature 1") # label the second feature axis.
plt.show() # display the region plot.

▶ What you'll see: each cluster lies inside a broad linear decision region for its class.

👀 Takeaway: softmax regression is multiclass linear classification with probabilistic outputs.

### Easy 5 — Evaluate on held-out examples

**Goal.** Compare training and validation loss, because the source context warns that the training number alone is not enough for generalization. We build it in 4 steps.

In [ ]:
X_train_e5 = np.array([[2.0, 0.2], [1.7, -0.1], [-0.2, 1.8], [0.1, 2.1], [-1.8, -1.1], [-2.1, -0.8]]) # training points.
y_train_e5 = np.array([0, 0, 1, 1, 2, 2]) # training labels.
X_val_e5 = np.array([[1.8, 0.4], [0.0, 1.7], [-1.7, -0.6]]) # held-out validation-like points.
y_val_e5 = np.array([0, 1, 2]) # held-out labels.
print("train/val sizes:", len(y_train_e5), len(y_val_e5)) # inspect split sizes.

In [ ]:
W_e5 = np.zeros((2, 3)) # initialize weights.
b_e5 = np.zeros(3) # initialize biases.
for step_e5 in range(300): # train on the training split only.
    Z_e5 = X_train_e5 @ W_e5 + b_e5 # compute train scores.
    P_e5 = np.exp(Z_e5 - np.max(Z_e5, axis=1, keepdims=True)) # stable exponentials.
    P_e5 = P_e5 / np.sum(P_e5, axis=1, keepdims=True) # train probabilities.
    G_e5 = (P_e5 - np.eye(3)[y_train_e5]) / len(y_train_e5) # average gradients.
    W_e5 -= 0.4 * (X_train_e5.T @ G_e5) # update weights.
    b_e5 -= 0.4 * np.sum(G_e5, axis=0) # update biases.
print("training done") # confirm optimization completed.

In [ ]:
def probs_e5(X): # define a tiny local probability helper for this example.
    Z = X @ W_e5 + b_e5 # compute scores.
    E = np.exp(Z - np.max(Z, axis=1, keepdims=True)) # stabilize then exponentiate.
    return E / np.sum(E, axis=1, keepdims=True) # normalize rows.
train_p_e5 = probs_e5(X_train_e5) # compute train probabilities.
val_p_e5 = probs_e5(X_val_e5) # compute validation probabilities.
train_loss_e5 = float(-np.mean(np.log(train_p_e5[np.arange(len(y_train_e5)), y_train_e5]))) # compute train loss.
val_loss_e5 = float(-np.mean(np.log(val_p_e5[np.arange(len(y_val_e5)), y_val_e5]))) # compute validation loss.
print("train loss:", round(train_loss_e5, 3), "validation loss:", round(val_loss_e5, 3)) # inspect both losses.
assert val_loss_e5 < 0.2 # verify the held-out toy points are handled well.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact generalization plot.
plt.bar(["train", "validation"], [train_loss_e5, val_loss_e5], color=["teal", "orange"]) # compare split losses.
plt.title("Easy 5: train vs validation loss") # title the plot.
plt.ylabel("cross-entropy") # label the loss axis.
plt.show() # display the chart.

▶ What you'll see: both losses are small, but the validation bar is the one that checks behavior on unseen points.

👀 Takeaway: validation loss tests whether a good training fit survived contact with future-like data.

## 🔴 Advanced

### Advanced 1 — Add L2 regularization to the objective

**Goal.** Train with a weight penalty, because regularization is the recurring lever for controlling flexibility. We build it in 4 steps.

In [ ]:
X_a1 = np.array([[2.0, 0.2], [1.7, -0.1], [-0.2, 1.8], [0.1, 2.1], [-1.8, -1.1], [-2.1, -0.8]]) # training points.
y_a1 = np.array([0, 0, 1, 1, 2, 2]) # labels.
lams_a1 = np.array([0.0, 0.05, 0.2, 1.0]) # regularization strengths to compare.
final_losses_a1 = [] # store cross-entropy plus penalty for each lambda.
weight_norms_a1 = [] # store learned weight sizes.
print("lambda grid:", lams_a1) # inspect the sweep values.

In [ ]:
for lam_a1 in lams_a1: # train one model per regularization strength.
    W_a1 = np.zeros((2, 3)) # reset weights.
    b_a1 = np.zeros(3) # reset biases.
    for step_a1 in range(300): # run batch gradient descent.
        Z_a1 = X_a1 @ W_a1 + b_a1 # compute scores.
        P_a1 = np.exp(Z_a1 - np.max(Z_a1, axis=1, keepdims=True)) # stable exponentials.
        P_a1 = P_a1 / np.sum(P_a1, axis=1, keepdims=True) # probabilities.
        G_a1 = (P_a1 - np.eye(3)[y_a1]) / len(y_a1) # score gradients.
        W_a1 -= 0.4 * (X_a1.T @ G_a1 + lam_a1 * W_a1) # update weights with L2 shrinkage.
        b_a1 -= 0.4 * np.sum(G_a1, axis=0) # update biases without L2 penalty.
    ce_a1 = float(-np.mean(np.log(P_a1[np.arange(len(y_a1)), y_a1]))) # compute final cross-entropy.
    obj_a1 = ce_a1 + 0.5 * lam_a1 * float(np.sum(W_a1 ** 2)) # add the L2 objective cost.
    final_losses_a1.append(obj_a1) # store penalized objective.
    weight_norms_a1.append(float(np.linalg.norm(W_a1))) # store weight size.
print("objectives:", np.round(final_losses_a1, 3)) # inspect objective values.
print("weight norms:", np.round(weight_norms_a1, 3)) # inspect shrinkage.

In [ ]:
assert weight_norms_a1[-1] < weight_norms_a1[0] # verify strong regularization shrinks weights.
print("strong lambda norm drop:", round(weight_norms_a1[0] - weight_norms_a1[-1], 3)) # inspect the shrinkage amount.

In [ ]:
plt.figure(figsize=(5, 3)) # create a compact regularization plot.
plt.plot(lams_a1, weight_norms_a1, marker="o", color="crimson") # show how weights shrink as lambda grows.
plt.title("Advanced 1: L2 shrinks class weights") # title the plot.
plt.xlabel("λ") # label the regularization axis.
plt.ylabel("||W||") # label the weight-size axis.
plt.show() # display the curve.

▶ What you'll see: larger λ produces smaller weight norms, which is the intended stability pressure.

👀 Takeaway: L2 regularization adds a cost that discourages brittle large weights.

### Advanced 2 — Inspect calibration with confidence bins

**Goal.** Compare predicted confidence with empirical accuracy, because softmax outputs are probabilities and should be inspected as probabilities. We build it in 4 steps.

In [ ]:
probs_a2 = np.array([0.92, 0.81, 0.74, 0.68, 0.57, 0.52, 0.43, 0.35]) # predicted confidence for eight examples.
correct_a2 = np.array([1, 1, 1, 0, 1, 0, 0, 0]) # whether the predicted class was correct.
bins_a2 = np.array([0.0, 0.5, 0.7, 0.85, 1.0]) # confidence bins.
print("confidences:", probs_a2) # inspect predictions before binning.

In [ ]:
bin_conf_a2 = [] # store average confidence per nonempty bin.
bin_acc_a2 = [] # store empirical accuracy per nonempty bin.
for lo_a2, hi_a2 in zip(bins_a2[:-1], bins_a2[1:]): # loop over confidence intervals.
    m_a2 = (probs_a2 >= lo_a2) & (probs_a2 < hi_a2) if hi_a2 < 1.0 else (probs_a2 >= lo_a2) & (probs_a2 <= hi_a2) # select bin members.
    if np.any(m_a2): # keep nonempty bins only.
        bin_conf_a2.append(float(np.mean(probs_a2[m_a2]))) # average confidence.
        bin_acc_a2.append(float(np.mean(correct_a2[m_a2]))) # average correctness.
print("bin confidence:", np.round(bin_conf_a2, 3)) # inspect confidence bins.
print("bin accuracy:", np.round(bin_acc_a2, 3)) # inspect empirical accuracy bins.

In [ ]:
cal_error_a2 = float(np.mean(np.abs(np.array(bin_conf_a2) - np.array(bin_acc_a2)))) # compute a tiny mean calibration gap.
print("mean calibration gap:", round(cal_error_a2, 3)) # inspect probability reliability.
assert round(cal_error_a2, 3) == 0.238 # verify the worked calibration number.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact reliability plot.
plt.plot([0, 1], [0, 1], color="black", linestyle="--", label="perfect") # show ideal calibration.
plt.scatter(bin_conf_a2, bin_acc_a2, color="purple", s=80) # plot observed bins.
plt.title("Advanced 2: confidence vs accuracy") # title the plot.
plt.xlabel("mean predicted confidence") # label x-axis.
plt.ylabel("empirical accuracy") # label y-axis.
plt.legend() # show ideal-line label.
plt.show() # display the calibration plot.

▶ What you'll see: bins near the diagonal are calibrated; gaps show over- or under-confidence.

👀 Takeaway: a softmax probability is useful only if its confidence scale is checked, not just its argmax.

### Advanced 3 — Trace a regularization-selection tradeoff

**Goal.** Choose λ using validation loss plus cost, because the lesson's decision arithmetic ranks full scores rather than raw fit alone. We build it in 4 steps.

In [ ]:
train_loss_a3 = np.array([0.090, 0.110, 0.150, 0.240]) # hypothetical training losses for four λ settings.
val_loss_a3 = np.array([0.310, 0.240, 0.220, 0.260]) # validation losses show overfit then underfit.
complexity_cost_a3 = np.array([0.120, 0.080, 0.050, 0.030]) # extra cost decreases as λ stabilizes the model.
lams_a3 = np.array([0.0, 0.05, 0.2, 1.0]) # λ labels for the settings.
print("validation losses:", val_loss_a3) # inspect validation quality before cost.

In [ ]:
decision_score_a3 = val_loss_a3 + complexity_cost_a3 # combine validation behavior with method cost.
best_a3 = int(np.argmin(decision_score_a3)) # select the lowest full score.
print("decision scores:", np.round(decision_score_a3, 3)) # inspect full scores.
print("best lambda:", lams_a3[best_a3]) # inspect selected λ.
assert round(float(decision_score_a3[best_a3]), 3) == 0.270 # verify the selected score.

In [ ]:
gap_to_flexible_a3 = float(decision_score_a3[0] - decision_score_a3[best_a3]) # compare no-regularization to selected score.
print("gap versus λ=0:", round(gap_to_flexible_a3, 3)) # inspect evidence for stabilization.
assert round(gap_to_flexible_a3, 3) == 0.160 # verify the gap calculation.

In [ ]:
plt.figure(figsize=(5, 3)) # create a compact selection plot.
plt.plot(lams_a3, val_loss_a3, marker="o", label="validation loss") # show validation-only curve.
plt.plot(lams_a3, decision_score_a3, marker="s", label="validation + cost") # show full decision score.
plt.axvline(lams_a3[best_a3], color="red", linestyle="--", label="selected λ") # mark selected λ.
plt.title("Advanced 3: choose by full score") # title the plot.
plt.xlabel("λ") # label regularization strength.
plt.ylabel("lower is better") # label score scale.
plt.legend() # show curve labels.
plt.show() # display the selection curve.

▶ What you'll see: the best validation-only and full-score choices can differ when cost is included.

👀 Takeaway: the model-selection scale must match the objective you actually intend to optimize.

### Advanced 4 — Show why large learning rates overshoot

**Goal.** Compare two gradient-descent step sizes, because softmax training can become unstable when updates are too aggressive. We build it in 4 steps.

In [ ]:
X_a4 = np.array([[2.0, 0.2], [1.7, -0.1], [-0.2, 1.8], [0.1, 2.1], [-1.8, -1.1], [-2.1, -0.8]]) # training data.
y_a4 = np.array([0, 0, 1, 1, 2, 2]) # labels.
rates_a4 = [0.4, 5.0] # compare a stable and intentionally large learning rate.
curves_a4 = [] # store loss curves.
print("learning rates:", rates_a4) # inspect the comparison.

In [ ]:
for eta_a4 in rates_a4: # run one training loop per learning rate.
    W_a4 = np.zeros((2, 3)) # reset weights.
    b_a4 = np.zeros(3) # reset biases.
    losses_a4 = [] # store this run's losses.
    for step_a4 in range(40): # run a short training sequence.
        Z_a4 = X_a4 @ W_a4 + b_a4 # compute scores.
        P_a4 = np.exp(Z_a4 - np.max(Z_a4, axis=1, keepdims=True)) # stable exponentials.
        P_a4 = P_a4 / np.sum(P_a4, axis=1, keepdims=True) # probabilities.
        loss_a4 = float(-np.mean(np.log(P_a4[np.arange(len(y_a4)), y_a4]))) # compute current loss.
        losses_a4.append(loss_a4) # record the loss.
        G_a4 = (P_a4 - np.eye(3)[y_a4]) / len(y_a4) # average score gradients.
        W_a4 -= eta_a4 * (X_a4.T @ G_a4) # update weights.
        b_a4 -= eta_a4 * np.sum(G_a4, axis=0) # update biases.
    curves_a4.append(losses_a4) # store the finished curve.
print("final losses:", [round(c[-1], 3) for c in curves_a4]) # inspect both outcomes.

In [ ]:
stable_final_a4 = curves_a4[0][-1] # final loss for stable rate.
large_max_a4 = max(curves_a4[1]) # worst loss reached by large rate.
print("stable final:", round(stable_final_a4, 3), "large max:", round(large_max_a4, 3)) # inspect stability.
assert stable_final_a4 < 0.1 # verify the modest rate learns.

In [ ]:
plt.figure(figsize=(5, 3)) # create a compact learning-rate plot.
plt.plot(curves_a4[0], label="η=0.4", color="teal") # plot stable rate.
plt.plot(curves_a4[1], label="η=5.0", color="crimson") # plot aggressive rate.
plt.yscale("log") # use log scale so both curves fit.
plt.title("Advanced 4: learning-rate stability") # title the plot.
plt.xlabel("step") # label optimization steps.
plt.ylabel("cross-entropy, log scale") # label loss scale.
plt.legend() # show labels.
plt.show() # display the comparison.

▶ What you'll see: the moderate step drops smoothly, while the large step can jump before settling or oscillating.

👀 Takeaway: gradient direction is useful only when the step size is small enough to respect the local approximation.

### Advanced 5 — Test shift-invariance and temperature

**Goal.** Inspect two score transformations, because softmax is invariant to common shifts but sensitive to temperature scaling. We build it in 4 steps.

In [ ]:
z_a5 = np.array([1.1, 0.6, -0.4]) # define base scores.
shifted_a5 = z_a5 + 100.0 # add the same constant to every class score.
temps_a5 = np.array([0.5, 1.0, 2.0]) # lower temperature sharpens; higher temperature smooths.
print("base scores:", z_a5) # inspect base scores.

In [ ]:
E_base_a5 = np.exp(z_a5 - np.max(z_a5)) # stable exponentials for base scores.
p_base_a5 = E_base_a5 / np.sum(E_base_a5) # base probabilities.
E_shift_a5 = np.exp(shifted_a5 - np.max(shifted_a5)) # stable exponentials after common shift.
p_shift_a5 = E_shift_a5 / np.sum(E_shift_a5) # shifted probabilities.
print("base p:", np.round(p_base_a5, 3)) # inspect base softmax.
print("shifted p:", np.round(p_shift_a5, 3)) # inspect shifted softmax.
assert np.allclose(p_base_a5, p_shift_a5) # verify shift-invariance.

In [ ]:
P_temp_a5 = [] # store softmax probabilities by temperature.
for T_a5 in temps_a5: # evaluate each temperature.
    scaled_a5 = z_a5 / T_a5 # divide scores by temperature.
    E_a5 = np.exp(scaled_a5 - np.max(scaled_a5)) # stable exponentials.
    P_temp_a5.append(E_a5 / np.sum(E_a5)) # normalize and store.
P_temp_a5 = np.array(P_temp_a5) # convert to matrix for inspection and plotting.
print("temperature probabilities:\n", np.round(P_temp_a5, 3)) # inspect sharpening and smoothing.
assert P_temp_a5[0, 0] > P_temp_a5[1, 0] > P_temp_a5[2, 0] # verify lower temperature is sharper for the top class.

In [ ]:
plt.figure(figsize=(5, 3)) # create a compact temperature plot.
for row_a5, T_a5 in zip(P_temp_a5, temps_a5): # plot one curve per temperature.
    plt.plot([0, 1, 2], row_a5, marker="o", label=f"T={T_a5}") # show probabilities across classes.
plt.xticks([0, 1, 2], ["c0", "c1", "c2"]) # label class positions.
plt.title("Advanced 5: temperature changes confidence") # title the plot.
plt.ylabel("probability") # label probability scale.
plt.legend() # show temperature labels.
plt.show() # display the chart.

▶ What you'll see: adding 100 changes nothing, while low temperature concentrates probability on the top class.

👀 Takeaway: softmax ignores common score shifts but temperature scaling changes confidence by stretching or shrinking score gaps.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Softmax regression compares class scores by exponentiating and normalizing them into probabilities.

Softmax regression extends the logistic idea to several classes while keeping one normalized probability scale. The lesson's arithmetic checks that the average loss, cost, and validation gap remain the unit of comparison.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_diabetes
from sklearn.datasets import make_blobs
from sklearn.datasets import make_classification
from sklearn.datasets import make_moons
from sklearn.datasets import make_regression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.linear_model import HuberRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import PoissonRegressor
from sklearn.linear_model import RANSACRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

np.random.seed(7)

def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...].

    All X are 2-D float feature matrices, y integer labels, so one classifier runs unchanged
    across every rung (the 'watch it scale' story). Rungs get harder: clean+separable -> real
    high-dimensional. D1 is hand-built and fully inspectable.
    """
    rungs = []

    # D1 — four hand-placed 2-D points, 2 classes, clearly separable.
    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    # D2 — clean, well-separated Gaussian blobs.
    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    # D3 — non-linear, overlapping two-moons with noise.
    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    # D4 — real: Wine, 13 features, 3 classes.
    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    # D5 — real, harder: Breast Cancer, 30 features, class imbalance.
    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", bc.data, bc.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def logistic_baseline(x_tr, y_tr, x_te):
    """Default classifier used to demonstrate a ladder end to end."""
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)


def lesson_score(losses, cost, alternative):
    losses = np.asarray(losses, dtype=float)
    raw = round(float(losses.mean()), 3)
    score = round(raw + cost, 3)
    gap = round(alternative - score, 3)
    return {
        "losses": losses,
        "raw": raw,
        "cost": cost,
        "score": score,
        "alternative": alternative,
        "gap": gap,
    }


def binary_logistic_train(x_tr, y_tr, lr=0.2, steps=900, l2=0.02):
    X = np.column_stack([np.ones(x_tr.shape[0]), x_tr])
    weights = np.zeros(X.shape[1])
    y = y_tr.astype(float)
    for step in range(steps):
        logits = X @ weights
        probs = 1.0 / (1.0 + np.exp(-logits))
        grad = X.T @ (probs - y) / y.size
        grad[1:] = grad[1:] + l2 * weights[1:]
        weights = weights - lr * grad
    return weights


def binary_logistic_predict(weights, x_te):
    X = np.column_stack([np.ones(x_te.shape[0]), x_te])
    probs = 1.0 / (1.0 + np.exp(-(X @ weights)))
    return (probs >= 0.5).astype(int)


def softmax_train(x_tr, y_tr, lr=0.15, steps=1100, l2=0.01):
    classes = np.unique(y_tr)
    mapping = {label: idx for idx, label in enumerate(classes)}
    y_idx = np.array([mapping[label] for label in y_tr])
    X = np.column_stack([np.ones(x_tr.shape[0]), x_tr])
    W = np.zeros((X.shape[1], classes.size))
    Y = np.eye(classes.size)[y_idx]
    for step in range(steps):
        logits = X @ W
        logits = logits - logits.max(axis=1, keepdims=True)
        exp_scores = np.exp(logits)
        probs = exp_scores / exp_scores.sum(axis=1, keepdims=True)
        grad = X.T @ (probs - Y) / X.shape[0]
        grad[1:, :] = grad[1:, :] + l2 * W[1:, :]
        W = W - lr * grad
    return W, classes


def softmax_predict(model, x_te):
    W, classes = model
    X = np.column_stack([np.ones(x_te.shape[0]), x_te])
    logits = X @ W
    return classes[np.argmax(logits, axis=1)]


def glm_predict(x_tr, y_tr, x_te):
    classes = np.unique(y_tr)
    if classes.size == 2:
        weights = binary_logistic_train(x_tr, y_tr)
        return binary_logistic_predict(weights, x_te)
    model = softmax_train(x_tr, y_tr)
    return softmax_predict(model, x_te)


def lda_qda_predict(x_tr, y_tr, x_te, mode="lda"):
    classes, counts = np.unique(y_tr, return_counts=True)
    if x_tr.shape[0] <= classes.size or counts.min() < 2:
        model = GaussianNB(var_smoothing=1e-8)
    elif mode == "qda":
        model = QuadraticDiscriminantAnalysis(reg_param=0.08)
    else:
        model = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def gda_predict(x_tr, y_tr, x_te):
    model = GaussianNB(var_smoothing=1e-8)
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def sklearn_logistic_predict(x_tr, y_tr, x_te):
    model = LogisticRegression(max_iter=2500)
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def classifier_metrics(rungs, predictor):
    rows = []
    for level, item in enumerate(rungs, start=1):
        name, X, y = item
        accuracy = clf_accuracy(predictor, X, y)
        rows.append({"level": level, "name": name, "accuracy": float(accuracy)})
    return rows


def plot_classifier_summary(rungs, rows, predictor):
    fig, axes = plt.subplots(2, 3, figsize=(13, 7))
    axes = axes.ravel()
    for ax, item, row in zip(axes[:5], rungs, rows):
        name, X, y = item
        x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
        scaler = StandardScaler()
        x_tr_s = scaler.fit_transform(x_tr)
        x_te_s = scaler.transform(x_te)
        preds = predictor(x_tr_s, y_tr, x_te_s)
        ax.scatter(x_te_s[:, 0], x_te_s[:, 1], c=preds, s=14, cmap="viridis", alpha=0.8)
        ax.set_title(f"D{row['level']} acc={row['accuracy']:.2f}")
        ax.set_xlabel("feature 0")
        ax.set_ylabel("feature 1")
    axes[5].plot([row["level"] for row in rows], [row["accuracy"] for row in rows], marker="o")
    axes[5].set_ylim(0.0, 1.05)
    axes[5].set_title("Accuracy vs ladder rung")
    axes[5].set_xlabel("D1 to D5")
    axes[5].set_ylabel("held-out accuracy")
    plt.tight_layout()
    plt.show()

def logistic_regression_method(losses=None, cost=0.070, alternative=0.394):
    if losses is None:
        losses = np.array([0.235, 0.109, 0.471])
    return lesson_score(losses, cost, alternative)


def softmax_multinomial_regression_method(losses=None, cost=0.080, alternative=0.401):
    if losses is None:
        losses = np.array([0.246, 0.122, 0.488])
    return lesson_score(losses, cost, alternative)


def generalized_linear_models_method(losses=None, cost=0.090, alternative=0.429):
    if losses is None:
        losses = np.array([0.257, 0.135, 0.505])
    return lesson_score(losses, cost, alternative)


def linear_quadratic_discriminant_analysis_method(losses=None, cost=0.100, alternative=0.457):
    if losses is None:
        losses = np.array([0.268, 0.148, 0.522])
    return lesson_score(losses, cost, alternative)


def gaussian_discriminant_analysis_method(losses=None, cost=0.050, alternative=0.361):
    if losses is None:
        losses = np.array([0.180, 0.070, 0.539])
    return lesson_score(losses, cost, alternative)

## The concept, built once (D1)

The lesson formula is

$$p_k=\frac{e^{z_k}}{\sum_j e^{z_j}}$$

For D1, the verified per-example losses are 0.246, 0.122, 0.488. The empirical risk is the average, and the model-selection score is that raw term plus the lesson cost.

In [ ]:
result = softmax_multinomial_regression_method()
print(result)
assert np.isclose(result["raw"], 0.285)
assert np.isclose(result["score"], 0.365)
assert np.isclose(result["gap"], 0.036)

The exact arithmetic is $R_S=(0.246, 0.122, 0.488)/3=0.285$, then $score=R_S+0.080=0.365$. The tempting alternative is 0.401, so the validation gap is $0.401-0.365=0.036$.

In [ ]:
stable_score = 0.80 * result["score"]
relative_gap = result["gap"] / result["alternative"]
print(f"stable={stable_score:.3f} relative_gap={relative_gap:.3f}")
assert stable_score < result["score"]
assert relative_gap > 0.0

## The dataset ladder

The same method now runs on D1 through D5. The printed preview shows shape, class balance or target scale, and a small sample before any fitting.

In [ ]:
rungs = clf_ladder()
for level, item in enumerate(rungs, start=1):
    name, X, y = item
    labels, counts = np.unique(y, return_counts=True)
    class_info = dict(zip(labels.tolist(), counts.tolist()))
    print(f"D{level}: {name} X={X.shape} classes={class_info}")
    print("sample X", np.round(X[:3, : min(3, X.shape[1])], 3))
    print("sample y", y[:3])

## Run the same method across D1-D5

The single headline metric for this lesson is accuracy. Classification topics also print the shared logistic baseline for a no-special-skill comparison.

In [ ]:
predictor = lambda x_tr, y_tr, x_te: softmax_predict(softmax_train(x_tr, y_tr), x_te)
rows = classifier_metrics(rungs, predictor)
print("rung | accuracy | logistic baseline | dataset")
for row, item in zip(rows, rungs):
    name, X, y = item
    baseline = clf_accuracy(logistic_baseline, X, y)
    print(f"D{row['level']} | {row['accuracy']:.3f} | {baseline:.3f} | {row['name']}")
assert len(rows) == 5
assert all(0.0 <= row["accuracy"] <= 1.0 for row in rows)

## Results visualization

The closing figure has two parts: small multiples for each rung and one summary curve from D1 to D5.

In [ ]:
plot_classifier_summary(rungs, rows, predictor)

## Pitfall on the hardest rung

Pitfall: optimizing the raw term and forgetting the cost. The wrong check looks only at the raw D5 metric; the fix restores the lesson's cost and gap before selecting a winner.

In [ ]:
name, X, y = rungs[-1]
x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
scaler = StandardScaler()
x_tr = scaler.fit_transform(x_tr)
x_te = scaler.transform(x_te)
main_preds = predictor(x_tr, y_tr, x_te)
base_preds = logistic_baseline(x_tr, y_tr, x_te)
main_acc = accuracy_score(y_te, main_preds)
base_acc = accuracy_score(y_te, base_preds)
lesson = softmax_multinomial_regression_method()
raw_only = 1.0 - main_acc
full_score = raw_only + lesson["cost"]
alt_score = (1.0 - base_acc) + lesson["alternative"]
print(f"Raw-only D5 error={raw_only:.3f} baseline_error={1.0 - base_acc:.3f}")
print(f"Full score with lesson cost={full_score:.3f} alternative score={alt_score:.3f}")
print(f"Lesson cost={lesson['cost']:.3f} gap={lesson['gap']:.3f}")
print(f"Macro-F1 sanity={f1_score(y_te, main_preds, average='macro'):.3f}")
assert lesson["gap"] > 0.0
assert 0.0 <= full_score

## Evaluate it + Practice

- Compare the headline metric with the no-skill or logistic baseline before claiming improvement.
- Run a cheap sanity check: shuffled labels should damage accuracy, and injected outliers should make robust regression matter.
- Ablation: turn off the key idea, such as Huber clipping, softmax normalization, the GLM link, covariance modeling, or Bayes priors; the D5 metric should usually drop.
- Failure signals include unstable validation gaps, wildly different scales, singular covariance warnings, or a D5 score that only wins before cost is included.

Practice 1: change the cost term and recompute the decision score.

Practice 2: rerun D5 after removing one informative feature group and compare the metric.

Practice 3: create a shuffled-label baseline and explain why it should fail.